# Synthetische Bauteile – Pipeline-Validierung

Vergleicht die eingebrachten Ground-Truth-Transformationen aus dem
synthetischen Datensatz mit den von der Pipeline gemessenen Werten.

## Methodik

Für jeden Testfall:
1. Ground Truth aus `synthetic_metadata.csv`
2. Pipeline-Ergebnisse aus `data/outputs/<name>/subtraction_report.json`
3. Fehler pro Freiheitsgrad berechnen und aggregieren

## Vorzeichen-Konventionen

Die Pipeline liefert `T_rel = T_B · T_A⁻¹`. Weil nur Werkstück B
transformiert wurde (A = Identität), ist T_rel die Inverse der eingebrachten
Transformation. Zusätzlich invertiert der Generator ty vor der Anwendung
(User-Konvention: positive ty = Spaltöffnung).

Aus dem Batch-Log:

| DOF     | GT-Vorzeichen  | Erwartetes Vorzeichen des gemessenen Wertes |
|---------|----------------|---------------------------------------------|
| tx_mm   | +              | – (T_rel-Inversion)                         |
| ty_mm   | +              | + (Generator + T_rel doppelt invertiert)    |
| tz_mm   | +              | – (T_rel-Inversion)                         |
| rx_deg  | ±              | ∓ (T_rel-Inversion)                         |
| ry_deg  | +              | – (T_rel-Inversion)                         |
| rz_deg  | +              | – (T_rel-Inversion)                         |

Wenn die Konvention stimmt, muss `expected = SIGN · GT ≈ measured`.


## 1. Konfiguration

In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SYNTH_DIR = Path("../data/raw/synthetic_scans")
OUTPUTS_DIR = Path("../data/outputs")

# Vorzeichen-Konventionen: expected_measurement = SIGN * GT
SIGN = {
    "tx_mm": -1,
    "ty_mm": +1,
    "tz_mm": -1,
    "rx_deg": -1,
    "ry_deg": -1,
    "rz_deg": -1,
}

# Farb-Konvention
COLOR_OK = "#43A047"       # grün
COLOR_WARN = "#FB8C00"     # orange
COLOR_FAIL = "#E53935"     # rot

## 2. Ground Truth + Ergebnisse einlesen

In [ ]:
df_gt = pd.read_csv(SYNTH_DIR / "synthetic_metadata.csv")
print(f"Ground Truth: {len(df_gt)} Testfälle")
print(f"Kategorien:")
for cat, n in df_gt["category"].value_counts().items():
    print(f"  {cat:35} {n:>3}")

In [ ]:
def load_result(case_name: str):
    """Lädt subtraction_report.json und gibt gemessene Werte zurück."""
    report_path = OUTPUTS_DIR / case_name / "subtraction_report.json"
    if not report_path.exists():
        return None

    with open(report_path) as f:
        report = json.load(f)

    reg = report.get("registration", {})
    dev = report.get("deviation", {})
    result = {
        "reg_residual_mm": reg.get("final_residual"),
        "overall_in_tolerance_rate": dev.get("overall_in_tolerance_rate"),
    }

    # ComponentRegistration – liegt direkt unter deviation
    comp_reg = dev.get("component_registration")
    if comp_reg is not None:
        translation = comp_reg.get("translation_mm", {})
        rotation = comp_reg.get("rotation_deg", {})
        result["measured_tx_mm"] = translation.get("x")
        result["measured_ty_mm"] = translation.get("y")
        result["measured_tz_mm"] = translation.get("z")
        result["measured_rx_deg"] = rotation.get("x")
        result["measured_ry_deg"] = rotation.get("y")
        result["measured_rz_deg"] = rotation.get("z")
        result["residual_a_mm"] = comp_reg.get("residual_a_mm")
        result["residual_b_mm"] = comp_reg.get("residual_b_mm")

    # PointDistance-Aggregate stehen aktuell nicht in der JSON (nur im
    # per_region_metrics für Label 0 als Näherung)
    label_0 = dev.get("per_region_metrics", {}).get("0", {})
    result["pd_mean_abs_mm"] = label_0.get("mean_abs")
    result["pd_rms_mm"] = label_0.get("rms")
    result["pd_max_abs_mm"] = label_0.get("max_abs")
    result["pd_p95_mm"] = label_0.get("p95")
    result["pd_in_tol_rate"] = label_0.get("in_tolerance_rate")

    # VoxelDeviation
    vox = dev.get("voxel_deviation")
    if vox is not None:
        counts = vox.get("counts", [])
        mean_abs = vox.get("mean_abs", [])
        tolerance = dev.get("tolerance_mm", 0.25)
        result["vox_total"] = len(counts)
        result["vox_out_of_tol"] = sum(1 for m in mean_abs if abs(m) > tolerance)

    return result

In [ ]:
rows = []
missing = []
for _, r in df_gt.iterrows():
    case_name = r["filename"].replace(".ply", "")
    res = load_result(case_name)
    if res is None:
        missing.append(case_name)
        continue
    combined = {**r.to_dict(), **res, "case_name": case_name}
    rows.append(combined)

df = pd.DataFrame(rows)

## 3. Erwartete gemessene Werte berechnen

Wendet die Vorzeichen-Konvention auf die Ground Truth an, damit die
Zahlen direkt vergleichbar mit den Pipeline-Werten sind.

In [ ]:
print(type(df) if 'df' in dir() else "df existiert nicht")

In [ ]:
for col in ["tx_mm", "ty_mm", "tz_mm", "rx_deg", "ry_deg", "rz_deg"]:
    df[f"expected_{col}"] = SIGN[col] * df[col]

# Fehler = gemessen - erwartet (in absoluten Zahlen und relativ)
for col in ["tx_mm", "ty_mm", "tz_mm", "rx_deg", "ry_deg", "rz_deg"]:
    df[f"err_{col}"] = df[f"measured_{col}"] - df[f"expected_{col}"]
    df[f"err_abs_{col}"] = df[f"err_{col}"].abs()

In [ ]:
# Debug: einen einzelnen Case laden und schauen
test_case = "T_Y_+01.500mm"
report_path = OUTPUTS_DIR / test_case / "subtraction_report.json"
print(f"Pfad existiert: {report_path.exists()}")
print(f"Absoluter Pfad: {report_path.absolute()}")

if report_path.exists():
    import json
    with open(report_path) as f:
        report = json.load(f)
    print(f"Keys top-level: {list(report.keys())}")
    print(f"Keys in deviation: {list(report.get('deviation', {}).keys())}")
    print(f"component_registration vorhanden: {'component_registration' in report.get('deviation', {})}")
    
    # Und was load_result damit macht:
    result = load_result(test_case)
    print(f"\nload_result Rückgabe: {result}")

## 4. Übersichtstabelle

Vergleich Ground Truth vs. Pipeline pro Bauteil.
Kritische Zeilen zeigen große Abweichungen.

In [ ]:
cols = [
    "case_name", "category",
    "tx_mm", "ty_mm", "tz_mm", "rx_deg", "ry_deg", "rz_deg",
    "measured_tx_mm", "measured_ty_mm", "measured_tz_mm",
    "measured_rx_deg", "measured_ry_deg", "measured_rz_deg",
    "reg_residual_mm",
]
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 20)

with pd.option_context("display.float_format", "{:+.3f}".format):
    print(df[cols].to_string(index=False))

## 5. Scatter-Plots pro Freiheitsgrad

Jeder Testfall = ein Punkt. Perfekte Übereinstimmung liegt auf der
Diagonalen (y = x). Abweichung von der Diagonale = Messfehler.

Für die Translation: **X-Achse ist erwartet (GT · SIGN), Y-Achse ist gemessen.**

In [ ]:
dofs = [
    ("tx_mm", "Translation X (mm)"),
    ("ty_mm", "Translation Y (mm)"),
    ("tz_mm", "Translation Z (mm)"),
    ("rx_deg", "Rotation X (°)"),
    ("ry_deg", "Rotation Y (°)"),
    ("rz_deg", "Rotation Z (°)"),
]

fig, axes = plt.subplots(2, 3, figsize=(16, 10), dpi=120)
for ax, (col, label) in zip(axes.flat, dofs):
    expected = df[f"expected_{col}"]
    measured = df[f"measured_{col}"]

    ax.scatter(expected, measured, s=25, alpha=0.6, color="#1976D2")

    # Diagonale y = x
    lo, hi = min(expected.min(), measured.min()), max(expected.max(), measured.max())
    span = hi - lo
    lo, hi = lo - span * 0.05, hi + span * 0.05
    ax.plot([lo, hi], [lo, hi], "--", color="gray", alpha=0.6, label="y = x (perfekt)")
    ax.set_xlim(lo, hi)
    ax.set_ylim(lo, hi)

    err_max = df[f"err_abs_{col}"].max()
    err_mean = df[f"err_abs_{col}"].mean()
    ax.set_title(f"{label}\n|Fehler|: mean={err_mean:.3f}, max={err_max:.3f}")
    ax.set_xlabel(f"Erwartet (SIGN · GT)")
    ax.set_ylabel(f"Gemessen (Pipeline)")
    ax.grid(alpha=0.3)
    ax.legend(loc="upper left", fontsize=8, framealpha=0.9)

plt.tight_layout()
plt.show()

## 6. Fehler-Statistik pro Kategorie

Zeigt Median und Max des Fehlerbetrags je Fehlerkategorie und
Freiheitsgrad. Große Werte bedeuten: dieser DOF wird bei dieser
Fehlerklasse nicht sauber rekonstruiert.

In [ ]:
summary = (
    df.groupby("category")[[f"err_abs_{c}" for c, _ in dofs]]
      .agg(["mean", "max"])
)
summary.columns = [
    f"{c.replace('err_abs_','')}_{stat}" for c, stat in summary.columns
]
with pd.option_context("display.float_format", "{:.3f}".format):
    print(summary.to_string())

## 7. PointDistance und VoxelDeviation

Diese Metriken skalieren mit der Gesamt-Abweichung des Bauteils – bei
großen Transformationen sollten sie größer sein als bei kleinen.

In [ ]:
# Als Größe eines Fehlers: euklidische Norm über alle DOF (normiert)
df["gt_translation_norm"] = np.sqrt(df["tx_mm"]**2 + df["ty_mm"]**2 + df["tz_mm"]**2)
df["gt_rotation_norm"] = np.sqrt(df["rx_deg"]**2 + df["ry_deg"]**2 + df["rz_deg"]**2)
df["gt_severity"] = df["gt_translation_norm"] + df["gt_rotation_norm"]

fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=120)

ax = axes[0]
sc = ax.scatter(df["gt_severity"], df["pd_mean_abs_mm"],
                c=df["reg_residual_mm"], cmap="viridis", s=30, alpha=0.7)
plt.colorbar(sc, ax=ax, label="Reg-Residuum (mm)")
ax.set_xlabel("GT Severity (Translation-Norm + Rotation-Norm)")
ax.set_ylabel("PointDistance |d|_mean (mm)")
ax.set_title("PointDistance vs. GT-Severity")
ax.grid(alpha=0.3)

ax = axes[1]
sc = ax.scatter(df["gt_severity"], df["vox_out_of_tol"],
                c=df["reg_residual_mm"], cmap="viridis", s=30, alpha=0.7)
plt.colorbar(sc, ax=ax, label="Reg-Residuum (mm)")
ax.set_xlabel("GT Severity")
ax.set_ylabel("Anzahl Voxel > Toleranz")
ax.set_title("VoxelDeviation vs. GT-Severity")
ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

## 8. Sonderfälle: Auffälligkeiten identifizieren

Zeigt die zehn Bauteile mit dem größten summierten Fehler.

In [ ]:
df["total_err"] = sum(df[f"err_abs_{c}"] for c, _ in dofs)

top_offenders = df.nlargest(10, "total_err")[
    ["case_name", "category", "total_err"] + [f"err_{c}" for c, _ in dofs]
]
print("Top 10 Testfälle mit größten Abweichungen:")
with pd.option_context("display.float_format", "{:+.3f}".format):
    print(top_offenders.to_string(index=False))

### 8.1 X-Translation

Bekanntes Problem: entlang der Naht-Längsachse ist die Geometrie
zu uniform, ICP hat wenig Halt.

In [ ]:
tx_only = df[df["category"] == "translation_x"].sort_values("tx_mm")
print("Translation X (nur X-Verschiebung, andere DOF = 0):")
cols_show = ["case_name", "tx_mm", "expected_tx_mm", "measured_tx_mm", "err_tx_mm"]
with pd.option_context("display.float_format", "{:+.3f}".format):
    print(tx_only[cols_show].to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5), dpi=120)
ax.plot(tx_only["tx_mm"], tx_only["measured_tx_mm"],
        "o-", color="#1976D2", label="Gemessen")
ax.plot(tx_only["tx_mm"], tx_only["expected_tx_mm"],
        "--", color="gray", label="Erwartet (−GT)")
ax.set_xlabel("Ground Truth tx_mm")
ax.set_ylabel("Wert")
ax.set_title("X-Translation: erwartet vs. gemessen")
ax.legend()
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 8.2 R_Y induziert Δz

Rotation um Y sollte nur `rot_y` verändern. Wenn dabei auch `Δz`
merklich abweicht, liegt das am Rotationsmittelpunkt außerhalb des
Werkstück-Schwerpunkts.

In [ ]:
ry_only = df[df["category"] == "rotation_y"].sort_values("ry_deg")
print("R_Y – gemessene Werte für ry, tz und ty:")
cols_show = ["case_name", "ry_deg", "measured_ry_deg",
             "measured_tx_mm", "measured_ty_mm", "measured_tz_mm"]
with pd.option_context("display.float_format", "{:+.3f}".format):
    print(ry_only[cols_show].to_string(index=False))

## 9. Zusammenfassung

Zusammenfassung Fehlerbeträge über alle Testfälle.

In [ ]:
print(f"{'DOF':<10} {'mean(|err|)':>12} {'p95(|err|)':>12} {'max(|err|)':>12}")
print("-" * 50)
for col, label in dofs:
    err = df[f"err_abs_{col}"]
    print(f"{col:<10} {err.mean():>12.4f} {np.percentile(err, 95):>12.4f} {err.max():>12.4f}")